<a href="https://colab.research.google.com/github/ALX-UFG-AKCIT-GEN-AI-2026/tcc/blob/main/tcc/bpmn-workflow-generator/training/colab/Fluxo_BPM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers
!pip install torch
!pip install lxml

In [ ]:
!pip install -U bitsandbytes accelerate transformers -q

In [ ]:
import os
import gdown
url = "https://huggingface.co/alexfdealmeida/qwen25-coder-7b-tcc/resolve/main/Qwen2.5-Coder-7B-Instruct.zip?download=true"
destino = "/content/Qwen2.5-Coder-7B-Instruct.zip"
gdown.download(url, destino, quiet=False)
print(f"Arquivo salvo em: {destino}")

In [ ]:
import zipfile
with zipfile.ZipFile('/content/Qwen2.5-Coder-7B-Instruct.zip', 'r') as zip_ref:
    zip_ref.extractall('/')

In [ ]:
import torch
import json
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
modelo = "/content/Qwen2.5-Coder-7B-Instruct"

# configuração de quantização
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(modelo)

# modelo quantizado
model = AutoModelForCausalLM.from_pretrained(
    modelo,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

In [ ]:
print(model.config._name_or_path)

In [ ]:
print(model.config.num_hidden_layers)
print(model.config.hidden_size)

In [ ]:
print(torch.cuda.get_device_name(0))

In [ ]:
print(torch.cuda.get_device_properties(0).total_memory / 1024**3)

TRATAMENTO CRIAÇÃO DO ARQUIVO BPMN 2.0


In [ ]:
import json
import os
from lxml import etree

class MotorGeradorBPMN:
    def __init__(self, dados_formulario):
        self.metadata = dados_formulario.get("metadata", {})
        self.nome_processo = self.metadata.get("nome", "Processo Customizado")
        self.id_processo = self.metadata.get("id", "processo_customizado")
        self.lanes = self.metadata.get("lanes", ["Setor Padrão"])
        self.elementos = dados_formulario.get("elementos", {})

        # Namespaces oficiais para compatibilidade com Camunda e Bizagi
        self.nsmap = {
            "bpmn": "http://www.omg.org/spec/BPMN/20100524/MODEL",
            "bpmndi": "http://www.omg.org/spec/BPMN/20100524/DI",
            "dc": "http://www.omg.org/spec/DD/20100524/DC",
            "di": "http://www.omg.org/spec/DD/20100524/DI"
        }

        # Raiz do documento XML
        self.root = etree.Element(f"{{{self.nsmap['bpmn']}}}definitions", nsmap=self.nsmap, id="Definitions_Form", targetNamespace="http://bpmn.io/schema/bpmn", exporter="MotorBPMN", exporterVersion="1.0")
        self.collaboration = etree.SubElement(self.root, f"{{{self.nsmap['bpmn']}}}collaboration", id="Collaboration_1")
        self.participant = etree.SubElement(self.collaboration, f"{{{self.nsmap['bpmn']}}}participant", id="Participant_1", name=self.nome_processo, processRef=self.id_processo)

        self.process = etree.SubElement(self.root, f"{{{self.nsmap['bpmn']}}}process", id=self.id_processo, isExecutable="true")
        self.lane_set = etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}laneSet", id="LaneSet_1")

        self.diagram = etree.SubElement(self.root, f"{{{self.nsmap['bpmndi']}}}BPMNDiagram", id="BPMNDiagram_1")
        self.plane = etree.SubElement(self.diagram, f"{{{self.nsmap['bpmndi']}}}BPMNPlane", id="BPMNPlane_1", bpmnElement="Collaboration_1")

        self.coordenadas = {}
        self.contador_fluxos = 0
        self.altura_raia = 280
        self.largura_total_pool = 2000

    def calcular_posicoes_grid(self):
        """ Calcula o layout ortogonal distribuindo o X sequencialmente e fixando o Y por Lane """
        altura_raia = self.altura_raia
        posicoes_raias = {lane: (i * altura_raia) + 150 for i, lane in enumerate(self.lanes)}

        id_inicio = next((k for k, v in self.elementos.items() if v["tipo"] == "start"), None)
        if not id_inicio:
            id_inicio = list(self.elementos.keys())[0] if self.elementos else None

        if not id_inicio:
            return

        # Algoritmo de Varredura em Largura (BFS) para ordenação de dependências visuais
        fila = [(id_inicio, 220)]
        visitados = set()

        while fila:
            id_no, x_atual = fila.pop(0)
            if id_no in visitados or id_no not in self.elementos:
                continue

            no = self.elementos[id_no]
            lane_atual = no.get("lane", self.lanes[0])
            centro_y = posicoes_raias.get(lane_atual, 150) + 70

            self.coordenadas[id_no] = (x_atual, centro_y)
            visitados.add(id_no)

            proximos = no.get("proximos", [])
            for prox in proximos:
                prox_id = prox if isinstance(prox, str) else prox.get("destino")
                fila.append((prox_id, x_atual + 240))

    def compilar_xml(self):
        self.calcular_posicoes_grid()
        altura_raia = self.altura_raia
        largura_total_pool = self.largura_total_pool

        # Desenha a Pool Geral
        shape_part = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id="Participant_1_di", bpmnElement="Participant_1", isHorizontal="true")
        etree.SubElement(shape_part, f"{{{self.nsmap['dc']}}}Bounds", x="120", y="100", width=str(largura_total_pool), height=str(altura_raia * len(self.lanes)))

        # Desenha as divisórias das Raias (Lanes)
        elementos_raias = {}
        for i, nome_lane in enumerate(self.lanes):
            id_lane = f"Lane_{i}"
            el_lane = etree.SubElement(self.lane_set, f"{{{self.nsmap['bpmn']}}}lane", id=id_lane, name=nome_lane)
            elementos_raias[nome_lane] = el_lane

            shape_lane = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_lane}_di", bpmnElement=id_lane, isHorizontal="true")
            etree.SubElement(shape_lane, f"{{{self.nsmap['dc']}}}Bounds", x="150", y=str((i * altura_raia) + 100), width=str(largura_total_pool - 30), height=str(altura_raia))

        # Criação dos Objetos de Fluxo (Nodes)
        for id_no, no in self.elementos.items():
            tipo = no["tipo"]
            texto = no["texto"]
            nome_lane = no.get("lane", self.lanes[0])
            x, y = self.coordenadas.get(id_no, (220, 150))

            if nome_lane in elementos_raias:
                etree.SubElement(elementos_raias[nome_lane], f"{{{self.nsmap['bpmn']}}}flowNodeRef").text = id_no

            if tipo == "start":
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}startEvent", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-18), width="36", height="36")

            elif tipo == "end":
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}endEvent", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-18), width="36", height="36")

            elif tipo in ["gateway_split", "gateway_merge"]:
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}exclusiveGateway", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-50), width="50", height="50")
            # criação do gateway_parallel
            elif tipo in ["gateway_parallel", "gateway_parallel_merge"]:
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}parallelGateway", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-40), width="50", height="50")

            elif tipo in ["gateway_inclusivo", "gateway_inclusivo_merge"]:
                # Alterado para 'inclusiveGateway' (simbolizado por um círculo)
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}inclusiveGateway", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-30), width="50", height="50")
            elif tipo in ["intermediate_event", "intermediate_event_merge"]:
                # Alterado para 'inclusiveGateway' (simbolizado por um círculo)
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}intermediateCatchEvent", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-25), width="20", height="20")



            elif tipo == "tarefa":
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}userTask", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-40), width="100", height="80")

            elif tipo == "subprocesso":
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}subProcess", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-40), width="100", height="80")

        # Conexão de Setas e Fluxos (Sequence Flows)
        for id_no, no in self.elementos.items():
            for prox in no.get("proximos", []):
                id_prox = prox if isinstance(prox, str) else prox.get("destino")
                texto_cond = "" if isinstance(prox, str) else prox.get("condicao", "")

                if id_prox in self.coordenadas:
                    self.contador_fluxos += 1
                    id_fluxo = f"Flow_{self.contador_fluxos}"

                    sf = etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}sequenceFlow", id=id_fluxo, sourceRef=id_no, targetRef=id_prox)
                    if texto_cond:
                        sf.set("name", texto_cond)

                    edge = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNEdge", id=f"{id_fluxo}_di", bpmnElement=id_fluxo)
                    x_orig, y_orig = self.coordenadas[id_no]
                    x_dest, y_dest = self.coordenadas[id_prox]

                    offset_orig = 100 if no["tipo"] == "tarefa" else (50 if "gateway" in no["tipo"] else 36)
                    etree.SubElement(edge, f"{{{self.nsmap['di']}}}waypoint", x=str(x_orig + offset_orig), y=str(y_orig))
                    etree.SubElement(edge, f"{{{self.nsmap['di']}}}waypoint", x=str(x_dest), y=str(y_dest))

    def validar(self):
      for id_no,no in self.elementos.items():

          if "texto" not in no:
              print("ERRO:", id_no, "sem texto")

          if "lane" not in no:
              print("ERRO:", id_no, "sem lane")

          if "proximos" not in no:
              print("ERRO:", id_no, "sem proximos")

          if " " in id_no:
              print("ERRO ID com espaço:", id_no)
    def salvar(self, nome_arquivo):

        self.compilar_xml()

        with open(nome_arquivo, "wb") as f:
            f.write(etree.tostring(self.root, pretty_print=True, xml_declaration=True, encoding="UTF-8"))
# =========================================================================
# DICIONÁRIO DE INTEGRALIZAÇÃO DE TODOS OS FORMULÁRIOS WEB DA INSTITUIÇÃO
# =========================================================================
formularios_completos = {

  "boletim_diario.bpmn": {
    "metadata": {
      "nome": "Boletim Diário",
      "id": "proc_boletim_diario",
      "lanes": [
        "Pauteiro",
        "Repórter",
        "Edição e revisão",
        "Envio e postagem"
      ]
    },
    "elementos": {
      "inicio_multiplo": {
        "tipo": "start",
        "texto": "Identificação matérias ou recebimento de demandas",
        "lane": "Pauteiro",
        "proximos": [
          {
            "destino": "gateway_viabilidade_pauta",
            "condicao": ""
          }
        ]
      },
      "gateway_viabilidade_pauta": {
        "tipo": "gateway_split",
        "texto": "É viável realizar a pauta?",
        "lane": "Pauteiro",
        "proximos": [
          {
            "destino": "frente_descartar_pauta",
            "condicao": "Não"
          },
          {
            "destino": "estabelecer_agenda_repor",
            "condicao": "Sim"
          }
        ]
      },
      "frente_descartar_pauta": {
        "tipo": "tarefa",
        "texto": "Descartar pauta e encerrar processo",
        "lane": "Pauteiro",
        "proximos": [
          {
            "destino": "fim_processo",
            "condicao": ""
          }
        ]
      },
      "estabelecer_agenda_repor": {
        "tipo": "tarefa",
        "texto": "Estabelecer agenda de pautas e encaminhar para o repórter",
        "lane": "Pauteiro",
        "proximos": [
          {
            "destino": "apuracao",
            "condicao": ""
          }
        ]
      },
      "apuracao": {
        "tipo": "tarefa",
        "texto": "Apurar",
        "lane": "Repórter",
        "proximos": [
          {
            "destino": "gateway_frente_1",
            "condicao": ""
          }
        ]
      },
      "gateway_frente_1": {
        "tipo": "gateway_split",
        "texto": "Frente 1: Descartar pauta e fim",
        "lane": "Repórter",
        "proximos": [
          {
            "destino": "frente_descartar_pauta",
            "condicao": ""
          },
          {
            "destino": "gateway_espera_tempo",
            "condicao": ""
          }
        ]
      },
      "gateway_espera_tempo": {
        "tipo": "gateway_split",
        "texto": "O tempo de espera é viável?",
        "lane": "Repórter",
        "proximos": [
          {
            "destino": "fim_processo",
            "condicao": "Não"
          },
          {
            "destino": "evento_espera_tempo",
            "condicao": "Sim"
          }
        ]
      },
      "evento_espera_tempo": {
        "tipo": "intermediate_event",
        "texto": "Esperar tempo necessário",
        "lane": "Repórter",
        "proximos": [
          {
            "destino": "frente_sistematizar_matérias",
            "condicao": ""
          }
        ]
      },
      "frente_sistematizar_matérias": {
        "tipo": "tarefa",
        "texto": "Sistematizar as matérias",
        "lane": "Edição e revisão",
        "proximos": [
          {
            "destino": "frente_editar_matéria",
            "condicao": ""
          }
        ]
      },
      "frente_editar_matéria": {
        "tipo": "tarefa",
        "texto": "Escrever matéria para a edição",
        "lane": "Edição e revisão",
        "proximos": [
          {
            "destino": "frente_editar_matéria",
            "condicao": ""
          }
        ]
      },
      "frente_editar_matéria_finalizado": {
        "tipo": "tarefa",
        "texto": "Cadastrar no portal e redes sociais",
        "lane": "Envio e postagem",
        "proximos": [
          {
            "destino": "fim_processo",
            "condicao": ""
          }
        ]
      },
      "fim_processo": {
        "tipo": "end",
        "texto": "Encerrar processo",
        "lane": "Todos",
        "proximos": []
      }
    }
  }


}
from difflib import get_close_matches
def corrigir_destinos(self, limite_similaridade=0.75):
    chaves = list(self['elementos'].keys())
    for id_no, no in self['elementos'].items():
        for prox in no['proximos']:
            if isinstance(prox, dict):
                destino = prox['destino']
                if destino == "":
                    continue
                else:
                    inicio = chaves.index(id_no)
                    chaves_list = chaves[inicio+1:]
                    print(chaves_list)
                    similar = get_close_matches(
                        destino,
                        chaves_list,
                        n=1,
                        cutoff=limite_similaridade
                    )
                    if similar and destino != similar[0]:
                       prox['destino'] = similar[0]
                       print("diferente")
    return  self
# Execução em lote (Batch processing)
if __name__ == "__main__":
    print("[*] Iniciando a compilação automatizada de processos por formulário...")
    for nome_arquivo, payload in formularios_completos.items():
        payload = corrigir_destinos(payload)
        motor = MotorGeradorBPMN(payload)
        #motor.validar()
        motor.salvar(nome_arquivo)
        print(f" [+] Arquivo '{nome_arquivo}' compilado com sucesso.")
    print("[*] Concluído! Todos os diagramas estão prontos para importação.")

TREINAMENTO DO MODELO

In [ ]:
import os
import gdown
url = "https://raw.githubusercontent.com/ALX-UFG-AKCIT-GEN-AI-2026/tcc/main/bpmn-workflow-generator/training/dataset/dataset_bpmn.jsonl"
destino = "/content/dataset_bpmn.jsonl"
gdown.download(url, destino, quiet=False)
print(f"Arquivo salvo em: {destino}")

CARREGANDO O JSON

In [ ]:
from datasets import load_dataset
dataset = load_dataset(
    "json",
    data_files="dataset_bpmn.jsonl",
    split="train"
)

print(dataset)
print(dataset[0])

In [ ]:
!pip install trl==0.18.0 -q

In [ ]:
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)
from trl import SFTTrainer


model.config.use_cache = False

model = prepare_model_for_kbit_training(model)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
"""
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM"
)"""

In [ ]:
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
  target_modules=[
          "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
training_args = SFTConfig(
    output_dir="./qwen-bpmn-lora",          # Pasta onde serão salvos checkpoints e o modelo final

    num_train_epochs=3,                     # Número de épocas (quantas vezes o modelo verá toda a base)

    per_device_train_batch_size=2,          # Quantidade de exemplos processados por GPU a cada passo

    gradient_accumulation_steps=4,          # Acumula gradientes por 4 passos antes de atualizar os pesos

    learning_rate=2e-4,                     # Taxa de aprendizado (0.0002)

    max_length=8192,                        # Tamanho máximo da sequência em tokens

    dataset_text_field="text",              # Nome da coluna que contém o texto do treinamento

    logging_steps=1,                        # Exibe métricas a cada passo

    save_steps=100,                         # Salva um checkpoint a cada 100 passos

    bf16=True,                              # Usa precisão BFloat16 (mais eficiente e estável)

    report_to="none"                        # Desativa integração com WandB, TensorBoard etc.
)

trainer = SFTTrainer(
    model=model,

    processing_class=tokenizer,

    train_dataset=dataset,

    args=training_args
    ,peft_config=lora_config
)

# ============================================================
# INICIAR TREINAMENTO
# ============================================================

trainer.train()

# ============================================================
# SALVAR ADAPTADOR LORA
# ============================================================

trainer.save_model(
    "./qwen-bpmn-lora-final"
)

tokenizer.save_pretrained(
    "./qwen-bpmn-lora-final"
)


print("Treinamento concluído!")

SALVANDO O MODELO COM FINE-TUNNING

In [ ]:
import shutil
base_name="qwen-bpmn-lora-final_2"
# Compacta a pasta "modelo"
shutil.make_archive(
    base_name=base_name,
    format="zip",
    root_dir="/content/qwen-bpmn-lora-final"
)

print(f"Arquivo criado: {base_name}.zip")

DESCOMPACTANDO MODELO SALVO

In [ ]:
import shutil

shutil.unpack_archive(
    "/content/qwen-bpmn-lora-final_2.zip",
    "/content/qwen-bpmn-lora-final_2"
)

print("Descompactado!")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

lora_path = "/content/qwen-bpmn-lora-final_2"
model2 = PeftModel.from_pretrained(model, lora_path)

In [ ]:
# pipeline
pipe = pipeline(
    "text-generation",
    model=model2,
    tokenizer=tokenizer
)

MODELO GRADIO UTILIZADO

In [ ]:
import gradio as gr
import json
import re
from datetime import datetime
from transformers import GenerationConfig
import torch
import gc

def corrigir_fechamento(texto):

    pilha = []
    fechamento = {
        "{": "}",
        "[": "]"
    }

    for c in texto:

        if c in "{[":
            pilha.append(c)

        elif c == "}":
            if pilha and pilha[-1] == "{":
                pilha.pop()

        elif c == "]":
            if pilha and pilha[-1] == "[":
                pilha.pop()

    # acrescenta os fechamentos na ordem inversa
    while pilha:
        texto += fechamento[pilha.pop()]

    return texto
def montar_prompt(texto):

    return f"""<s>[INST]
Você é um especialista em BPMN 2.0 e Engenharia de Dados.

Converta o texto fornecido para JSON BPMN válido, seguindo RIGOROSAMENTE as regras abaixo.

RETORNE APENAS O JSON.
NÃO explique.
NÃO escreva texto fora do JSON.
NÃO use Markdown.

REGRAS:

1. Use apenas os tipos permitidos:
   ✓ "start"
   ✓ "tarefa"
   ✓ "subprocesso"
   ✓ "gateway_split"
   ✓ "gateway_merge"
   ✓ "gateway_parallel"
   ✓ "gateway_parallel_merge"
   ✓ "gateway_inclusivo"
   ✓ "gateway_inclusivo_merge"
   ✓ "intermediate_event"
   ✓ "intermediate_event_merge"
   ✓ "end"

2. MAPEAMENTO SEMÂNTICO ESTRITO:
  ✓ Início do processo, gatilho inicial ou evento que inicia o fluxo → "start"
  ✓ Ação, atividade, execução, preenchimento, envio, análise, conferência ou qualquer trabalho realizado por um participante → "tarefa"
  ✓ Decisão, validação, aprovação, verificação, disponibilidade ou qualquer ponto do processo que resulte em dois ou mais caminhos alternativos → "gateway_split"
  ✓ Ponto de reunião de caminhos alternativos originados por um gateway_split (somente um caminho retorna) → "gateway_merge"
  ✓ Abertura de atividades executadas simultaneamente (AND) → "gateway_parallel"
  ✓ Fechamento de atividades paralelas obrigatórias (AND Join), aguardando todos os ramos concluírem → "gateway_parallel_merge"
  ✓ Abertura de fluxos opcionais, onde um ou mais caminhos podem ser executados simultaneamente (OR) → "gateway_inclusivo"
  ✓ Fechamento de fluxos opcionais (OR Join), aguardando apenas os ramos que realmente foram ativados→ "gateway_inclusivo_merge"
  ✓ Evento intermediário de espera, mensagem, tempo, sinal ou sincronização durante o fluxo → "intermediate_event"
  ✓ Evento intermediário utilizado para reunir ou sincronizar fluxos que convergem em um evento → "intermediate_event_merge"
  ✓ Processo composto por um fluxo interno detalhado → "subprocesso"
  ✓ Encerramento do processo ou término de um ramo  → "end"

3. Estrutura obrigatória do JSON:
{{
  "<nome_arquivo>.bpmn": {{
    "metadata": {{
      "nome": "",
      "id": "",
      "lanes": []
    }},
    "elementos": {{
        "": {{
        "tipo":"",
        "texto":"",
        "lane":"",
        "proximos":[{{"destino": "", "condicao": ""}}]
      }}
    }}
  }}
}}

CRITÉRIOS E EXEMPLOS OBRIGATÓRIOS PARA ESCOLHA DE GATEWAYS:
  1. gateway_split (Exclusivo / XOR):
    - QUANDO USAR: Quando ocorrer duas ou mais saídas com condições de escolha. (Ex: "Aprovado ou Reprovado?", "Sim ou Não?").
    - Exemplo:
      "gateway_aprovacao": {{
        "tipo": "gateway_split",
        "texto": "O documento foi aprovado?",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "tarefa_seguir", "condicao": "Sim"}},
          {{"destino": "tarefa_revisar", "condicao": "Não"}}
        ]
      }}

  2. gateway_parallel (Paralelo / AND):
    - QUANDO USAR: Quando ocorrer duas ou mais saídas sem condições de escolha.
    - Exemplo:
      "gateway_abrir_paralelo": {{
        "tipo": "gateway_parallel",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "realizar_inspecao", "condicao": "Frente Manutenção"}},
          {{"destino": "verificar_docs", "condicao": "Frente Documentação"}}
        ]
      }}

  3. gateway_inclusivo (Inclusivo / OR):
    - QUANDO USAR: Quando podem ocorrer um ou mais saídas opcionais e combinados com base em condições.
    - Exemplo:
      "gateway_inclusivo_ajustes": {{
        "tipo": "gateway_inclusivo",
        "texto": "Quais ajustes são necessários?",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "revisao_dados", "condicao": "Apontamentos estatísticos"}},
          {{"destino": "revisao_estilo", "condicao": "Apontamentos gramaticais"}}
        ]
      }}

  4. gateway_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_split.
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}
  5. gateway_parallel_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_parallel .
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_parallel_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}
  6. gateway_inclusivo_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_inclusivo.
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_inclusivo_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}

VALIDAÇÃO FINAL OBRIGATÓRIA ANTES DE GERAR:
✓ Não existem elementos órfãos (todo nó possui ao menos uma entrada, exceto o start).
✓ Todos os elementos citados em "proximos" existem no dicionário de elementos (sem chaves órfãs ou erradas).
✓ O campo "proximos" de "subprocesso" e "end" é exatamente [].
✓ Todo processo deve possuir pelo menos um elemento do tipo "end".
✓ Apenas gateways podem possuir múltiplos destinos.
✓ Nenhuma tarefa pode possuir dois ou mais elementos em "proximos".
✓ Toda pergunta (texto terminado com "?") deve ser representada por um "gateway_split", nunca por uma "tarefa".
TEXTO:
{texto}
[/INST]
"""

# ============================================================
# LIMPEZA DO JSON
# ============================================================

def extrair_json(texto):

    texto = texto.strip()

    match = re.search(r'\{.*\}', texto, re.DOTALL)

    if match:
        return match.group()

    return texto

# ============================================================
# GERA JSON BPMN
# ============================================================

def gerar_json_bpmn(descricao):
    print("Iniciando")

    prompt = montar_prompt(descricao)


    # =====================================
    # CHAMADA AO MODELO
    # =====================================
    # 1. Configuração focada em velocidade e rigidez estrutural
    gen_config = GenerationConfig(
        max_new_tokens=8048,  # Linha única gasta pouquíssimo token, 2048 sobra muito!
        do_sample=False,
        temperature=0.7,
        pad_token_id=pipe.tokenizer.eos_token_id,
    )

    # 2. Execução da Pipeline
    resp = pipe(prompt, generation_config=gen_config, return_full_text=False)
    resposta_bruta = resp[0]["generated_text"]
    print(resposta_bruta)
    string_limpa = resposta_bruta.strip()
    match = re.search(r"(\{.*\})", string_limpa, re.DOTALL)

    # Só avança se o Regex realmente capturou algo
    if match:
        json_linha_unica = match.group(1)

        try:
            # Converte para dicionário Python
            dados = json.loads(json_linha_unica)
            print("✅ JSON extraído e convertido com sucesso!")
            print(f"Nome do Processo Identificado: {dados}")
            return json.dumps(
                dados,
                ensure_ascii=False,
                indent=2
            )

        # Esse try/except protege seu código caso o JSON venha quebrado/cortado pelo modelo
        except json.JSONDecodeError as e:
            print("Tentativa de Correção")
            texto_corrigido = corrigir_fechamento(json_linha_unica)
            try:
                dados = json.loads(texto_corrigido)
                print("✅ JSON extraído e convertido com sucesso!")
                return json.dumps(
                dados,
                ensure_ascii=False,
                indent=2
            )
            except json.JSONDecodeError as e:
                print("❌ Falha crítica no Parseamento do JSON")
                print(f"Conteúdo bruto capturado: {resposta_bruta}")
                return (
                f"ERRO AO INTERPRETAR JSON\n\n"
                f"{str(e)}\n\n"
                f"RESPOSTA:\n{resposta_bruta}"
            )

    else:
        # Se caiu aqui, é porque a resposta do modelo não tinha chaves {}
        print("❌ Nenhum padrão JSON foi encontrado na resposta.")

# ============================================================
# GERA ARQUIVO JSON
# ============================================================

def salvar_json(json_texto):

    nome = (
        f"bpmn_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    )

    with open(nome, "w", encoding="utf-8") as f:
        f.write(json_texto)

    return nome

# ============================================================
# GERA ARQUIVO BPMN
# ============================================================

def salvar_bpmn(json_texto):
    try:
        json_texto = json.loads(json_texto)
        nome_arquivo = list(json_texto.keys())[0]
        estrutura = json_texto[nome_arquivo]
        estrutura = corrigir_destinos(estrutura)
        motor = MotorGeradorBPMN(estrutura)
        motor.salvar(nome_arquivo)
        return (nome_arquivo)

    except Exception as e:

        raise gr.Error(
            f"Erro ao gerar BPMN: {str(e)}"
        )

def limpar():
    # Limpa a memória cache do PyTorch
    torch.cuda.empty_cache()

    # Força o coletor de lixo do Python a liberar objetos não utilizados
    gc.collect()
    return "", "", None, None
# ============================================================
# INTERFACE
# ============================================================

with gr.Blocks(
    title="Gerador BPMN"
) as demo:

    gr.Markdown("# 🤖 Gerador Automatizado de Grafos BPMN com Robozito")

    descricao = gr.Textbox(
        label="Descrição do Processo",
        lines=20,
        placeholder="""
Exemplo:
Processo de Solicitação de Equipamento
O colaborador registra uma solicitação de equipamento no sistema.
O setor de TI analisa a solicitação para verificar se as informações estão completas.
Caso as informações não estejam completas, o setor de TI solicita correção ao colaborador. Após a correção, a solicitação deve retornar para análise.
Caso as informações estejam completas, o setor de TI verifica se o equipamento está disponível em estoque.
Se o equipamento estiver disponível, o equipamento é separado para entrega.
Se o equipamento não estiver disponível, o setor de compras realiza a aquisição do equipamento.
Após a separação ou aquisição, o gestor deve aprovar a entrega.
Caso o gestor não aprove a entrega, a solicitação é encerrada.
Caso o gestor aprove a entrega, o equipamento é entregue ao colaborador.
Após a entrega ou encerramento da solicitação, o processo é finalizado.
"""
    )


    with gr.Row():
      analisar = gr.Button(
        "Gerar JSON BPMN",
        variant="primary"
      )
      btn_limpar = gr.Button(
          "🗑️ Limpar"
      )
    with gr.Row():
      json_saida = gr.Code(
    label="JSON BPMN",
    language="json",
    lines=25,
    min_width=700,
    visible=True,
    value='{\n  "Gerador": "Gerador"\n}'
)

    with gr.Row():

        btn_json = gr.Button(
            "⚡Baixar JSON",
            variant="primary"
        )

        btn_bpmn = gr.Button(
            "⚡Gerar BPMN",
            variant="primary"
        )

    arquivo_json = gr.File(
        label="Arquivo JSON"
    )

    arquivo_bpmn = gr.File(
        label="Arquivo BPMN"
    )

    analisar.click(
        gerar_json_bpmn,
        inputs=descricao,
        outputs=json_saida
    )

    btn_json.click(
        salvar_json,
        inputs=json_saida,
        outputs=arquivo_json
    )

    btn_bpmn.click(
        salvar_bpmn,
        inputs=json_saida,
        outputs=arquivo_bpmn
    )

    btn_limpar.click(
    fn=limpar,
    inputs=[],
    outputs=[
        descricao,
        json_saida,
        arquivo_json,
        arquivo_bpmn
    ]
)

# ============================================================
# EXECUÇÃO
# ============================================================

demo.launch(
    share=True,
    debug=True
)

In [ ]:
import gradio as gr
import json
import re
from datetime import datetime
from transformers import GenerationConfig
import torch
import gc

def corrigir_fechamento(texto):

    pilha = []
    fechamento = {
        "{": "}",
        "[": "]"
    }

    for c in texto:

        if c in "{[":
            pilha.append(c)

        elif c == "}":
            if pilha and pilha[-1] == "{":
                pilha.pop()

        elif c == "]":
            if pilha and pilha[-1] == "[":
                pilha.pop()

    # acrescenta os fechamentos na ordem inversa
    while pilha:
        texto += fechamento[pilha.pop()]

    return texto
def montar_prompt(texto):

    return f"""<s>[INST]
Você é um especialista em BPMN 2.0 e Engenharia de Dados.

Converta o texto fornecido para JSON BPMN válido, seguindo RIGOROSAMENTE as regras abaixo.

RETORNE APENAS O JSON.
NÃO explique.
NÃO escreva texto fora do JSON.
NÃO use Markdown.

REGRAS:

1. Use apenas os tipos permitidos:
   ✓ "start"
   ✓ "tarefa"
   ✓ "subprocesso"
   ✓ "gateway_split"
   ✓ "gateway_merge"
   ✓ "gateway_parallel"
   ✓ "gateway_parallel_merge"
   ✓ "gateway_inclusivo"
   ✓ "gateway_inclusivo_merge"
   ✓ "intermediate_event"
   ✓ "intermediate_event_merge"
   ✓ "end"

2. MAPEAMENTO SEMÂNTICO ESTRITO:
  ✓ Início do processo, gatilho inicial ou evento que inicia o fluxo → "start"
  ✓ Ação, atividade, execução, preenchimento, envio, análise, conferência ou qualquer trabalho realizado por um participante → "tarefa"
  ✓ Decisão, validação, aprovação, verificação, disponibilidade ou qualquer ponto do processo que resulte em dois ou mais caminhos alternativos → "gateway_split"
  ✓ Ponto de reunião de caminhos alternativos originados por um gateway_split (somente um caminho retorna) → "gateway_merge"
  ✓ Abertura de atividades executadas simultaneamente (AND) → "gateway_parallel"
  ✓ Fechamento de atividades paralelas obrigatórias (AND Join), aguardando todos os ramos concluírem → "gateway_parallel_merge"
  ✓ Abertura de fluxos opcionais, onde um ou mais caminhos podem ser executados simultaneamente (OR) → "gateway_inclusivo"
  ✓ Fechamento de fluxos opcionais (OR Join), aguardando apenas os ramos que realmente foram ativados→ "gateway_inclusivo_merge"
  ✓ Evento intermediário de espera, mensagem, tempo, sinal ou sincronização durante o fluxo → "intermediate_event"
  ✓ Evento intermediário utilizado para reunir ou sincronizar fluxos que convergem em um evento → "intermediate_event_merge"
  ✓ Processo composto por um fluxo interno detalhado → "subprocesso"
  ✓ Encerramento do processo ou término de um ramo  → "end"

3. Estrutura obrigatória do JSON:
{{
  "<nome_arquivo>.bpmn": {{
    "metadata": {{
      "nome": "",
      "id": "",
      "lanes": []
    }},
    "elementos": {{
        "": {{
        "tipo":"",
        "texto":"",
        "lane":"",
        "proximos":[{{"destino": "", "condicao": ""}}]
      }}
    }}
  }}
}}

CRITÉRIOS E EXEMPLOS OBRIGATÓRIOS PARA ESCOLHA DE GATEWAYS:
  1. gateway_split (Exclusivo / XOR):
    - QUANDO USAR: Quando ocorrer duas ou mais saídas com condições de escolha. (Ex: "Aprovado ou Reprovado?", "Sim ou Não?").
    - Exemplo:
      "gateway_aprovacao": {{
        "tipo": "gateway_split",
        "texto": "O documento foi aprovado?",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "tarefa_seguir", "condicao": "Sim"}},
          {{"destino": "tarefa_revisar", "condicao": "Não"}}
        ]
      }}

  2. gateway_parallel (Paralelo / AND):
    - QUANDO USAR: Quando ocorrer duas ou mais saídas sem condições de escolha.
    - Exemplo:
      "gateway_abrir_paralelo": {{
        "tipo": "gateway_parallel",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "realizar_inspecao", "condicao": "Frente Manutenção"}},
          {{"destino": "verificar_docs", "condicao": "Frente Documentação"}}
        ]
      }}

  3. gateway_inclusivo (Inclusivo / OR):
    - QUANDO USAR: Quando podem ocorrer um ou mais saídas opcionais e combinados com base em condições.
    - Exemplo:
      "gateway_inclusivo_ajustes": {{
        "tipo": "gateway_inclusivo",
        "texto": "Quais ajustes são necessários?",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "revisao_dados", "condicao": "Apontamentos estatísticos"}},
          {{"destino": "revisao_estilo", "condicao": "Apontamentos gramaticais"}}
        ]
      }}

  4. gateway_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_split.
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}
  5. gateway_parallel_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_parallel .
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_parallel_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}
  6. gateway_inclusivo_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_inclusivo.
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_inclusivo_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}

VALIDAÇÃO FINAL OBRIGATÓRIA ANTES DE GERAR:
✓ Não existem elementos órfãos (todo nó possui ao menos uma entrada, exceto o start).
✓ Todos os elementos citados em "proximos" existem no dicionário de elementos (sem chaves órfãs ou erradas).
✓ O campo "proximos" de "subprocesso" e "end" é exatamente [].
✓ Todo processo deve possuir pelo menos um elemento do tipo "end".
✓ Apenas gateways podem possuir múltiplos destinos.
✓ Nenhuma tarefa pode possuir dois ou mais elementos em "proximos".
✓ Toda pergunta (texto terminado com "?") deve ser representada por um "gateway_split", nunca por uma "tarefa".
TEXTO:
{texto}
[/INST]
"""

# ============================================================
# LIMPEZA DO JSON
# ============================================================

def extrair_json(texto):

    texto = texto.strip()

    match = re.search(r'\{.*\}', texto, re.DOTALL)

    if match:
        return match.group()

    return texto

# ============================================================
# GERA JSON BPMN
# ============================================================

def gerar_json_bpmn(descricao):
    print("Iniciando")

    prompt = montar_prompt(descricao)


    # =====================================
    # CHAMADA AO MODELO
    # =====================================
    # 1. Configuração focada em velocidade e rigidez estrutural
    gen_config = GenerationConfig(
        max_new_tokens=8048,  # Linha única gasta pouquíssimo token, 2048 sobra muito!
        do_sample=False,
        temperature=0.7,
        pad_token_id=pipe.tokenizer.eos_token_id,
    )

    # 2. Execução da Pipeline
    resp = pipe(prompt, generation_config=gen_config, return_full_text=False)
    resposta_bruta = resp[0]["generated_text"]
    print(resposta_bruta)
    string_limpa = resposta_bruta.strip()
    match = re.search(r"(\{.*\})", string_limpa, re.DOTALL)

    # Só avança se o Regex realmente capturou algo
    if match:
        json_linha_unica = match.group(1)

        try:
            # Converte para dicionário Python
            dados = json.loads(json_linha_unica)
            print("✅ JSON extraído e convertido com sucesso!")
            print(f"Nome do Processo Identificado: {dados}")
            return json.dumps(
                dados,
                ensure_ascii=False,
                indent=2
            )

        # Esse try/except protege seu código caso o JSON venha quebrado/cortado pelo modelo
        except json.JSONDecodeError as e:
            print("Tentativa de Correção")
            texto_corrigido = corrigir_fechamento(json_linha_unica)
            try:
                dados = json.loads(texto_corrigido)
                print("✅ JSON extraído e convertido com sucesso!")
                return json.dumps(
                dados,
                ensure_ascii=False,
                indent=2
            )
            except json.JSONDecodeError as e:
                print("❌ Falha crítica no Parseamento do JSON")
                print(f"Conteúdo bruto capturado: {resposta_bruta}")
                return (
                f"ERRO AO INTERPRETAR JSON\n\n"
                f"{str(e)}\n\n"
                f"RESPOSTA:\n{resposta_bruta}"
            )

    else:
        # Se caiu aqui, é porque a resposta do modelo não tinha chaves {}
        print("❌ Nenhum padrão JSON foi encontrado na resposta.")

# ============================================================
# GERA ARQUIVO JSON
# ============================================================

def salvar_json(json_texto):

    nome = (
        f"bpmn_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    )

    with open(nome, "w", encoding="utf-8") as f:
        f.write(json_texto)

    return nome

# ============================================================
# GERA ARQUIVO BPMN
# ============================================================

def salvar_bpmn(json_texto):
    try:
        json_texto = json.loads(json_texto)
        nome_arquivo = list(json_texto.keys())[0]
        estrutura = json_texto[nome_arquivo]
        estrutura = corrigir_destinos(estrutura)
        motor = MotorGeradorBPMN(estrutura)
        motor.salvar(nome_arquivo)
        return (nome_arquivo)

    except Exception as e:

        raise gr.Error(
            f"Erro ao gerar BPMN: {str(e)}"
        )

def limpar():
    # Limpa a memória cache do PyTorch
    torch.cuda.empty_cache()

    # Força o coletor de lixo do Python a liberar objetos não utilizados
    gc.collect()
    return "", "", None, None
# ============================================================
# INTERFACE
# ============================================================

with gr.Blocks(
    title="Gerador BPMN"
) as demo:

    gr.Markdown("# 🤖 Gerador Automatizado de Grafos BPMN com Robozito")

    descricao = gr.Textbox(
        label="Descrição do Processo",
        lines=20,
        placeholder="""
Exemplo:
Processo de Solicitação de Equipamento
O colaborador registra uma solicitação de equipamento no sistema.
O setor de TI analisa a solicitação para verificar se as informações estão completas.
Caso as informações não estejam completas, o setor de TI solicita correção ao colaborador. Após a correção, a solicitação deve retornar para análise.
Caso as informações estejam completas, o setor de TI verifica se o equipamento está disponível em estoque.
Se o equipamento estiver disponível, o equipamento é separado para entrega.
Se o equipamento não estiver disponível, o setor de compras realiza a aquisição do equipamento.
Após a separação ou aquisição, o gestor deve aprovar a entrega.
Caso o gestor não aprove a entrega, a solicitação é encerrada.
Caso o gestor aprove a entrega, o equipamento é entregue ao colaborador.
Após a entrega ou encerramento da solicitação, o processo é finalizado.
"""
    )


    with gr.Row():
      analisar = gr.Button(
        "Gerar JSON BPMN",
        variant="primary"
      )
      btn_limpar = gr.Button(
          "🗑️ Limpar"
      )
    with gr.Row():
      json_saida = gr.Code(
    label="JSON BPMN",
    language="json",
    lines=25,
    min_width=700,
    visible=True,
    value='{\n  "Gerador": "Gerador"\n}'
)

    with gr.Row():

        btn_json = gr.Button(
            "⚡Baixar JSON",
            variant="primary"
        )

        btn_bpmn = gr.Button(
            "⚡Gerar BPMN",
            variant="primary"
        )

    arquivo_json = gr.File(
        label="Arquivo JSON"
    )

    arquivo_bpmn = gr.File(
        label="Arquivo BPMN"
    )

    analisar.click(
        gerar_json_bpmn,
        inputs=descricao,
        outputs=json_saida
    )

    btn_json.click(
        salvar_json,
        inputs=json_saida,
        outputs=arquivo_json
    )

    btn_bpmn.click(
        salvar_bpmn,
        inputs=json_saida,
        outputs=arquivo_bpmn
    )

    btn_limpar.click(
    fn=limpar,
    inputs=[],
    outputs=[
        descricao,
        json_saida,
        arquivo_json,
        arquivo_bpmn
    ]
)

# ============================================================
# EXECUÇÃO
# ============================================================

demo.launch(
    share=True,
    debug=True
)

TESTE COM MINISTRAL


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

modelo = "mistralai/Mistral-7B-Instruct-v0.3"

# Quantização 4 bits
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(modelo)

# Modelo
model = AutoModelForCausalLM.from_pretrained(
    modelo,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

Testes com outros modelos

In [ ]:
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Qwen-7B-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="./deepseek-bpmn",
        num_train_epochs=5,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        max_length=4096,
        logging_steps=5,
        save_steps=100,
        report_to="none",
    ),
)

In [ ]:
# ============================================================
# INICIAR TREINAMENTO
# ============================================================

trainer.train()



# ============================================================
# SALVAR ADAPTADOR LORA
# ============================================================

trainer.save_model(
    "./deepseek-bpmn-lora-final"
)

tokenizer.save_pretrained(
    "./deepseek-bpmn-lora-final"
)


print("Treinamento concluído!")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

lora_path = "/content/deepseek-bpmn-lora-final"
model2 = PeftModel.from_pretrained(model, lora_path)

In [ ]:
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
# pipeline
pipe = pipeline(
    "text-generation",
    model=model2,
    tokenizer=tokenizer
)

In [ ]:
import gradio as gr
import json
import re
from datetime import datetime
from transformers import GenerationConfig
import torch
import gc

def corrigir_fechamento(texto):

    pilha = []
    fechamento = {
        "{": "}",
        "[": "]"
    }

    for c in texto:

        if c in "{[":
            pilha.append(c)

        elif c == "}":
            if pilha and pilha[-1] == "{":
                pilha.pop()

        elif c == "]":
            if pilha and pilha[-1] == "[":
                pilha.pop()

    # acrescenta os fechamentos na ordem inversa
    while pilha:
        texto += fechamento[pilha.pop()]

    return texto
def montar_prompt(texto):

    return f"""<s>[INST]
Você é um especialista em BPMN 2.0 e Engenharia de Dados.

Converta o texto fornecido para JSON BPMN válido, seguindo RIGOROSAMENTE as regras abaixo.

RETORNE APENAS O JSON.
NÃO explique.
NÃO escreva texto fora do JSON.
NÃO use Markdown.

REGRAS:

1. Use apenas os tipos permitidos:
   ✓ "start"
   ✓ "tarefa"
   ✓ "subprocesso"
   ✓ "gateway_split"
   ✓ "gateway_merge"
   ✓ "gateway_parallel"
   ✓ "gateway_parallel_merge"
   ✓ "gateway_inclusivo"
   ✓ "gateway_inclusivo_merge"
   ✓ "intermediate_event"
   ✓ "intermediate_event_merge"
   ✓ "end"

2. MAPEAMENTO SEMÂNTICO ESTRITO:
  ✓ Início do processo, gatilho inicial ou evento que inicia o fluxo → "start"
  ✓ Ação, atividade, execução, preenchimento, envio, análise, conferência ou qualquer trabalho realizado por um participante → "tarefa"
  ✓ Decisão, validação, aprovação, verificação, disponibilidade ou qualquer ponto do processo que resulte em dois ou mais caminhos alternativos → "gateway_split"
  ✓ Ponto de reunião de caminhos alternativos originados por um gateway_split (somente um caminho retorna) → "gateway_merge"
  ✓ Abertura de atividades executadas simultaneamente (AND) → "gateway_parallel"
  ✓ Fechamento de atividades paralelas obrigatórias (AND Join), aguardando todos os ramos concluírem → "gateway_parallel_merge"
  ✓ Abertura de fluxos opcionais, onde um ou mais caminhos podem ser executados simultaneamente (OR) → "gateway_inclusivo"
  ✓ Fechamento de fluxos opcionais (OR Join), aguardando apenas os ramos que realmente foram ativados→ "gateway_inclusivo_merge"
  ✓ Evento intermediário de espera, mensagem, tempo, sinal ou sincronização durante o fluxo → "intermediate_event"
  ✓ Evento intermediário utilizado para reunir ou sincronizar fluxos que convergem em um evento → "intermediate_event_merge"
  ✓ Processo composto por um fluxo interno detalhado → "subprocesso"
  ✓ Encerramento do processo ou término de um ramo  → "end"

3. Estrutura obrigatória do JSON:
{{
  "<nome_arquivo>.bpmn": {{
    "metadata": {{
      "nome": "",
      "id": "",
      "lanes": [<Nome das Lanes>]
    }},
    "elementos": "<nome_elemento>": {{
        "tipo": "<Tipo elemento>",
        "texto":"<Texto Elemento>",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "<Destino elemento>", "condicao": "<Texto>"}}
        ]
      }}
  }}
}}

CRITÉRIOS E EXEMPLOS OBRIGATÓRIOS PARA ESCOLHA DE GATEWAYS:
  1. gateway_split (Exclusivo / XOR):
    - QUANDO USAR: Quando ocorrer duas ou mais saídas com condições de escolha. (Ex: "Aprovado ou Reprovado?", "Sim ou Não?").
    - Exemplo:
      "gateway_aprovacao": {{
        "tipo": "gateway_split",
        "texto": "O documento foi aprovado?",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "tarefa_seguir", "condicao": "Sim"}},
          {{"destino": "tarefa_revisar", "condicao": "Não"}}
        ]
      }}

  2. gateway_parallel (Paralelo / AND):
    - QUANDO USAR: Quando ocorrer duas ou mais saídas sem condições de escolha.
    - Exemplo:
      "gateway_abrir_paralelo": {{
        "tipo": "gateway_parallel",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "realizar_inspecao", "condicao": "Frente Manutenção"}},
          {{"destino": "verificar_docs", "condicao": "Frente Documentação"}}
        ]
      }}

  3. gateway_inclusivo (Inclusivo / OR):
    - QUANDO USAR: Quando podem ocorrer um ou mais saídas opcionais e combinados com base em condições.
    - Exemplo:
      "gateway_inclusivo_ajustes": {{
        "tipo": "gateway_inclusivo",
        "texto": "Quais ajustes são necessários?",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "revisao_dados", "condicao": "Apontamentos estatísticos"}},
          {{"destino": "revisao_estilo", "condicao": "Apontamentos gramaticais"}}
        ]
      }}

  4. gateway_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_split.
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}
  5. gateway_parallel_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_parallel .
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_parallel_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}
  6. gateway_inclusivo_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_inclusivo.
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_inclusivo_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}

ESTRUTURA PADRÃO OBRIGATÓRIA DOS ELEMENTOS:
Cada elemento dentro de "elementos" deve obrigatoriamente conter SOMENTE os seguintes atributos:
{{
  "tipo": "<Tipo elemento>",
  "texto": "<Descrição visual do elemento>",
  "lane": "<Nome obrigatório da Lane>",
  "proximos": []
}}

REGRAS DOS ATRIBUTOS:
✓ "tipo":
- Campo obrigatório.
- Deve utilizar exclusivamente um dos tipos permitidos.
- Nunca criar novos tipos.

✓ "texto":
- Campo obrigatório para TODOS os elementos.
- Nunca utilizar "nome".
- Nunca utilizar "descricao".
- O conteúdo representa o texto exibido no elemento BPMN.
- Para gateways sem rótulo, utilizar string vazia "".
- Para start e end, utilizar string vazia "" quando não houver descrição.

✓ "lane":
- Campo obrigatório para TODOS os elementos.
- Deve sempre conter uma das Lanes existentes em metadata.lanes.
- Nunca deixar vazio.
- Nunca criar uma Lane nova dentro dos elementos.

✓ "proximos":
- Campo obrigatório para TODOS os elementos.
- Deve sempre ser uma lista.
- Caso não exista próximo elemento, utilizar obrigatoriamente:
  "proximos": []


VALIDAÇÃO FINAL OBRIGATÓRIA ANTES DE GERAR:
✓ Não existem elementos órfãos (todo nó possui ao menos uma entrada, exceto o start).
✓ Todos os elementos citados em "proximos" existem no dicionário de elementos (sem chaves órfãs ou erradas).
✓ O campo "proximos" de "subprocesso" e "end" é exatamente [].
✓ Todo processo deve possuir pelo menos um elemento do tipo "end".
✓ Apenas gateways podem possuir múltiplos destinos.
✓ Nenhuma tarefa pode possuir dois ou mais elementos em "proximos".
✓ Toda pergunta (texto terminado com "?") deve ser representada por um "gateway_split", nunca por uma "tarefa".
✓  Nomes dos lanes são obrigatório
✓  Use apenas os tipos permitidos
✓  Nome do arquivo deve ser nesse formato <nome_arquivo>.bpmn

{texto}
[/INST]
"""

# ============================================================
# LIMPEZA DO JSON
# ============================================================

def extrair_json(texto):

    texto = texto.strip()

    match = re.search(r'\{.*\}', texto, re.DOTALL)

    if match:
        return match.group()

    return texto

# ============================================================
# GERA JSON BPMN
# ============================================================

def gerar_json_bpmn(descricao):
    print("Iniciando")

    prompt = montar_prompt(descricao)


    # =====================================
    # CHAMADA AO MODELO
    # =====================================
    # 1. Configuração focada em velocidade e rigidez estrutural
    gen_config = GenerationConfig(
        max_new_tokens=8048,  # Linha única gasta pouquíssimo token, 2048 sobra muito!
        do_sample=False,
        temperature=0.2,
        pad_token_id=pipe.tokenizer.eos_token_id,
    )

    # 2. Execução da Pipeline
    resp = pipe(prompt, generation_config=gen_config, return_full_text=False)
    resposta_bruta = resp[0]["generated_text"]
    print(resposta_bruta)
    string_limpa = resposta_bruta.strip()
    match = re.search(r"(\{.*\})", string_limpa, re.DOTALL)

    # Só avança se o Regex realmente capturou algo
    if match:
        json_linha_unica = match.group(1)

        try:
            # Converte para dicionário Python
            dados = json.loads(json_linha_unica)
            print("✅ JSON extraído e convertido com sucesso!")
            print(f"Nome do Processo Identificado: {dados}")
            return json.dumps(
                dados,
                ensure_ascii=False,
                indent=2
            )

        # Esse try/except protege seu código caso o JSON venha quebrado/cortado pelo modelo
        except json.JSONDecodeError as e:
            print("Tentativa de Correção")
            texto_corrigido = corrigir_fechamento(json_linha_unica)
            try:
                dados = json.loads(texto_corrigido)
                print("✅ JSON extraído e convertido com sucesso!")
                return json.dumps(
                dados,
                ensure_ascii=False,
                indent=2
            )
            except json.JSONDecodeError as e:
                print("❌ Falha crítica no Parseamento do JSON")
                print(f"Conteúdo bruto capturado: {resposta_bruta}")
                return (
                f"ERRO AO INTERPRETAR JSON\n\n"
                f"{str(e)}\n\n"
                f"RESPOSTA:\n{resposta_bruta}"
            )

    else:
        # Se caiu aqui, é porque a resposta do modelo não tinha chaves {}
        print("❌ Nenhum padrão JSON foi encontrado na resposta.")

# ============================================================
# GERA ARQUIVO JSON
# ============================================================

def salvar_json(json_texto):

    nome = (
        f"bpmn_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    )

    with open(nome, "w", encoding="utf-8") as f:
        f.write(json_texto)

    return nome

# ============================================================
# GERA ARQUIVO BPMN
# ============================================================

def salvar_bpmn(json_texto):
    try:
        json_texto = json.loads(json_texto)
        nome_arquivo = list(json_texto.keys())[0]
        estrutura = json_texto[nome_arquivo]
        estrutura = corrigir_destinos(estrutura)
        motor = MotorGeradorBPMN(estrutura)
        motor.salvar(nome_arquivo)
        return (nome_arquivo)

    except Exception as e:

        raise gr.Error(
            f"Erro ao gerar BPMN: {str(e)}"
        )

def limpar():
    # Limpa a memória cache do PyTorch
    torch.cuda.empty_cache()

    # Força o coletor de lixo do Python a liberar objetos não utilizados
    gc.collect()
    return "", "", None, None
# ============================================================
# INTERFACE
# ============================================================

with gr.Blocks(
    title="Gerador BPMN"
) as demo:

    gr.Markdown("# 🤖 Gerador Automatizado de Grafos BPMN com Robozito")

    descricao = gr.Textbox(
        label="Descrição do Processo",
        lines=20,
        placeholder="""
Exemplo:
Processo de Solicitação de Equipamento
O colaborador registra uma solicitação de equipamento no sistema.
O setor de TI analisa a solicitação para verificar se as informações estão completas.
Caso as informações não estejam completas, o setor de TI solicita correção ao colaborador. Após a correção, a solicitação deve retornar para análise.
Caso as informações estejam completas, o setor de TI verifica se o equipamento está disponível em estoque.
Se o equipamento estiver disponível, o equipamento é separado para entrega.
Se o equipamento não estiver disponível, o setor de compras realiza a aquisição do equipamento.
Após a separação ou aquisição, o gestor deve aprovar a entrega.
Caso o gestor não aprove a entrega, a solicitação é encerrada.
Caso o gestor aprove a entrega, o equipamento é entregue ao colaborador.
Após a entrega ou encerramento da solicitação, o processo é finalizado.
"""
    )


    with gr.Row():
      analisar = gr.Button(
        "Gerar JSON BPMN",
        variant="primary"
      )
      btn_limpar = gr.Button(
          "🗑️ Limpar"
      )
    with gr.Row():
      json_saida = gr.Code(
    label="JSON BPMN",
    language="json",
    lines=25,
    min_width=700,
    visible=True,
    value='{\n  "Gerador": "Gerador"\n}'
)

    with gr.Row():

        btn_json = gr.Button(
            "⚡Baixar JSON",
            variant="primary"
        )

        btn_bpmn = gr.Button(
            "⚡Gerar BPMN",
            variant="primary"
        )

    arquivo_json = gr.File(
        label="Arquivo JSON"
    )

    arquivo_bpmn = gr.File(
        label="Arquivo BPMN"
    )

    analisar.click(
        gerar_json_bpmn,
        inputs=descricao,
        outputs=json_saida
    )

    btn_json.click(
        salvar_json,
        inputs=json_saida,
        outputs=arquivo_json
    )

    btn_bpmn.click(
        salvar_bpmn,
        inputs=json_saida,
        outputs=arquivo_bpmn
    )

    btn_limpar.click(
    fn=limpar,
    inputs=[],
    outputs=[
        descricao,
        json_saida,
        arquivo_json,
        arquivo_bpmn
    ]
)

# ============================================================
# EXECUÇÃO
# ============================================================

demo.launch(
    share=True,
    debug=True
)

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

In [ ]:
from unsloth import FastModel
import torch

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None, # None for auto detection
    max_seq_length = 1024, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

In [ ]:
from unsloth import FastLanguageModel
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="./gemma-4-bpmn",
        num_train_epochs=10,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        max_length=4096,
        logging_steps=5,
        save_steps=100,
        report_to="none",
    ),
)

In [ ]:
# ============================================================
# INICIAR TREINAMENTO
# ============================================================

trainer.train()



# ============================================================
# SALVAR ADAPTADOR LORA
# ============================================================

trainer.save_model(
    "./gemma-bpmn-lora-final"
)

tokenizer.save_pretrained(
    "./gemma-bpmn-lora-final"
)


print("Treinamento concluído!")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

lora_path = "/content/gemma-bpmn-lora-final"
model2 = PeftModel.from_pretrained(model, lora_path)

In [ ]:
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
# pipeline
pipe = pipeline(
    "text-generation",
    model=model2,
    tokenizer=tokenizer
)

In [ ]:
import gradio as gr
import json
import re
from datetime import datetime
from transformers import GenerationConfig
import torch
import gc

def corrigir_fechamento(texto):

    pilha = []
    fechamento = {
        "{": "}",
        "[": "]"
    }

    for c in texto:

        if c in "{[":
            pilha.append(c)

        elif c == "}":
            if pilha and pilha[-1] == "{":
                pilha.pop()

        elif c == "]":
            if pilha and pilha[-1] == "[":
                pilha.pop()

    # acrescenta os fechamentos na ordem inversa
    while pilha:
        texto += fechamento[pilha.pop()]

    return texto
def montar_prompt(texto):

    return f"""<s>[INST]
Você é um especialista em BPMN 2.0 e Engenharia de Dados.

Converta o texto fornecido para JSON BPMN válido, seguindo RIGOROSAMENTE as regras abaixo.

RETORNE APENAS O JSON.
NÃO explique.
NÃO escreva texto fora do JSON.
NÃO use Markdown.

REGRAS:

1. Use apenas os tipos permitidos:
   ✓ "start"
   ✓ "tarefa"
   ✓ "subprocesso"
   ✓ "gateway_split"
   ✓ "gateway_merge"
   ✓ "gateway_parallel"
   ✓ "gateway_parallel_merge"
   ✓ "gateway_inclusivo"
   ✓ "gateway_inclusivo_merge"
   ✓ "intermediate_event"
   ✓ "intermediate_event_merge"
   ✓ "end"

2. MAPEAMENTO SEMÂNTICO ESTRITO:
  ✓ Início do processo, gatilho inicial ou evento que inicia o fluxo → "start"
  ✓ Ação, atividade, execução, preenchimento, envio, análise, conferência ou qualquer trabalho realizado por um participante → "tarefa"
  ✓ Decisão, validação, aprovação, verificação, disponibilidade ou qualquer ponto do processo que resulte em dois ou mais caminhos alternativos → "gateway_split"
  ✓ Ponto de reunião de caminhos alternativos originados por um gateway_split (somente um caminho retorna) → "gateway_merge"
  ✓ Abertura de atividades executadas simultaneamente (AND) → "gateway_parallel"
  ✓ Fechamento de atividades paralelas obrigatórias (AND Join), aguardando todos os ramos concluírem → "gateway_parallel_merge"
  ✓ Abertura de fluxos opcionais, onde um ou mais caminhos podem ser executados simultaneamente (OR) → "gateway_inclusivo"
  ✓ Fechamento de fluxos opcionais (OR Join), aguardando apenas os ramos que realmente foram ativados→ "gateway_inclusivo_merge"
  ✓ Evento intermediário de espera, mensagem, tempo, sinal ou sincronização durante o fluxo → "intermediate_event"
  ✓ Evento intermediário utilizado para reunir ou sincronizar fluxos que convergem em um evento → "intermediate_event_merge"
  ✓ Processo composto por um fluxo interno detalhado → "subprocesso"
  ✓ Encerramento do processo ou término de um ramo  → "end"

3. Estrutura obrigatória do JSON:
{{
  "<nome_arquivo>.bpmn": {{
    "metadata": {{
      "nome": "",
      "id": "",
      "lanes": [<Nome das Lanes>]
    }},
    "elementos": "<nome_elemento>": {{
        "tipo": "<Tipo elemento>",
        "texto":"<Texto Elemento>",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "<Destino elemento>", "condicao": "<Texto>"}}
        ]
      }}
  }}
}}

CRITÉRIOS E EXEMPLOS OBRIGATÓRIOS PARA ESCOLHA DE GATEWAYS:
  1. gateway_split (Exclusivo / XOR):
    - QUANDO USAR: Quando ocorrer duas ou mais saídas com condições de escolha. (Ex: "Aprovado ou Reprovado?", "Sim ou Não?").
    - Exemplo:
      "gateway_aprovacao": {{
        "tipo": "gateway_split",
        "texto": "O documento foi aprovado?",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "tarefa_seguir", "condicao": "Sim"}},
          {{"destino": "tarefa_revisar", "condicao": "Não"}}
        ]
      }}

  2. gateway_parallel (Paralelo / AND):
    - QUANDO USAR: Quando ocorrer duas ou mais saídas sem condições de escolha.
    - Exemplo:
      "gateway_abrir_paralelo": {{
        "tipo": "gateway_parallel",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "realizar_inspecao", "condicao": "Frente Manutenção"}},
          {{"destino": "verificar_docs", "condicao": "Frente Documentação"}}
        ]
      }}

  3. gateway_inclusivo (Inclusivo / OR):
    - QUANDO USAR: Quando podem ocorrer um ou mais saídas opcionais e combinados com base em condições.
    - Exemplo:
      "gateway_inclusivo_ajustes": {{
        "tipo": "gateway_inclusivo",
        "texto": "Quais ajustes são necessários?",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "revisao_dados", "condicao": "Apontamentos estatísticos"}},
          {{"destino": "revisao_estilo", "condicao": "Apontamentos gramaticais"}}
        ]
      }}

  4. gateway_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_split.
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}
  5. gateway_parallel_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_parallel .
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_parallel_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}
  6. gateway_inclusivo_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_inclusivo.
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_inclusivo_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}

ESTRUTURA PADRÃO OBRIGATÓRIA DOS ELEMENTOS:
Cada elemento dentro de "elementos" deve obrigatoriamente conter SOMENTE os seguintes atributos:
{{
  "tipo": "<Tipo elemento>",
  "texto": "<Descrição visual do elemento>",
  "lane": "<Nome obrigatório da Lane>",
  "proximos": []
}}

REGRAS DOS ATRIBUTOS:
✓ "tipo":
- Campo obrigatório.
- Deve utilizar exclusivamente um dos tipos permitidos.
- Nunca criar novos tipos.

✓ "texto":
- Campo obrigatório para TODOS os elementos.
- Nunca utilizar "nome".
- Nunca utilizar "descricao".
- O conteúdo representa o texto exibido no elemento BPMN.
- Para gateways sem rótulo, utilizar string vazia "".
- Para start e end, utilizar string vazia "" quando não houver descrição.

✓ "lane":
- Campo obrigatório para TODOS os elementos.
- Deve sempre conter uma das Lanes existentes em metadata.lanes.
- Nunca deixar vazio.
- Nunca criar uma Lane nova dentro dos elementos.

✓ "proximos":
- Campo obrigatório para TODOS os elementos.
- Deve sempre ser uma lista.
- Caso não exista próximo elemento, utilizar obrigatoriamente:
  "proximos": []


VALIDAÇÃO FINAL OBRIGATÓRIA ANTES DE GERAR:
✓ Não existem elementos órfãos (todo nó possui ao menos uma entrada, exceto o start).
✓ Todos os elementos citados em "proximos" existem no dicionário de elementos (sem chaves órfãs ou erradas).
✓ O campo "proximos" de "subprocesso" e "end" é exatamente [].
✓ Todo processo deve possuir pelo menos um elemento do tipo "end".
✓ Apenas gateways podem possuir múltiplos destinos.
✓ Nenhuma tarefa pode possuir dois ou mais elementos em "proximos".
✓ Toda pergunta (texto terminado com "?") deve ser representada por um "gateway_split", nunca por uma "tarefa".
✓  Nomes dos lanes são obrigatório
✓  Use apenas os tipos permitidos
✓  Nome do arquivo deve ser nesse formato <nome_arquivo>.bpmn

{texto}
[/INST]
"""

# ============================================================
# LIMPEZA DO JSON
# ============================================================

def extrair_json(texto):

    texto = texto.strip()

    match = re.search(r'\{.*\}', texto, re.DOTALL)

    if match:
        return match.group()

    return texto

# ============================================================
# GERA JSON BPMN
# ============================================================

def gerar_json_bpmn(descricao):
    print("Iniciando")
    # 1. Montagem e validação preventiva do prompt
    prompt = montar_prompt(descricao)
    if not prompt or not isinstance(prompt, str):
        prompt = str(prompt) if prompt is not None else ""

    # 2. Aplicação obrigatória do Chat Template (Resolve o problema de não retornar nada)
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        messages = [{"role": "user", "content": prompt}]
        prompt_formatado = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    else:
        prompt_formatado = prompt

    # 3. Contorno seguro para evitar o erro do unsloth_zoo / Gemma no tokenizador
    base_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
    try:
        inputs = tokenizer(text=[prompt_formatado], return_tensors="pt").to("cuda")
    except TypeError:
        inputs = base_tokenizer(prompt_formatado, return_tensors="pt").to("cuda")

    # 4. Geração direta estruturada
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True
    )

    # 5. Decodificação isolando apenas a resposta gerada (removendo o prompt de entrada)
    input_len = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_len:]
    resposta_bruta = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    print(resposta_bruta)


    string_limpa = resposta_bruta.strip()
    match = re.search(r"(\{.*\})", string_limpa, re.DOTALL)

    # Só avança se o Regex realmente capturou algo
    if match:
        json_linha_unica = match.group(1)

        try:
            # Converte para dicionário Python
            dados = json.loads(json_linha_unica)
            print("✅ JSON extraído e convertido com sucesso!")
            print(f"Nome do Processo Identificado: {dados}")
            return json.dumps(
                dados,
                ensure_ascii=False,
                indent=2
            )

        # Esse try/except protege seu código caso o JSON venha quebrado/cortado pelo modelo
        except json.JSONDecodeError as e:
            print("Tentativa de Correção")
            texto_corrigido = corrigir_fechamento(json_linha_unica)
            try:
                dados = json.loads(texto_corrigido)
                print("✅ JSON extraído e convertido com sucesso!")
                return json.dumps(
                dados,
                ensure_ascii=False,
                indent=2
            )
            except json.JSONDecodeError as e:
                print("❌ Falha crítica no Parseamento do JSON")
                print(f"Conteúdo bruto capturado: {resposta_bruta}")
                return (
                f"ERRO AO INTERPRETAR JSON\n\n"
                f"{str(e)}\n\n"
                f"RESPOSTA:\n{resposta_bruta}"
            )

    else:
        # Se caiu aqui, é porque a resposta do modelo não tinha chaves {}
        print("❌ Nenhum padrão JSON foi encontrado na resposta.")

# ============================================================
# GERA ARQUIVO JSON
# ============================================================

def salvar_json(json_texto):

    nome = (
        f"bpmn_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    )

    with open(nome, "w", encoding="utf-8") as f:
        f.write(json_texto)

    return nome

# ============================================================
# GERA ARQUIVO BPMN
# ============================================================

def salvar_bpmn(json_texto):
    try:
        json_texto = json.loads(json_texto)
        nome_arquivo = list(json_texto.keys())[0]
        estrutura = json_texto[nome_arquivo]
        estrutura = corrigir_destinos(estrutura)
        motor = MotorGeradorBPMN(estrutura)
        motor.salvar(nome_arquivo)
        return (nome_arquivo)

    except Exception as e:

        raise gr.Error(
            f"Erro ao gerar BPMN: {str(e)}"
        )

def limpar():
    # Limpa a memória cache do PyTorch
    torch.cuda.empty_cache()

    # Força o coletor de lixo do Python a liberar objetos não utilizados
    gc.collect()
    return "", "", None, None
# ============================================================
# INTERFACE
# ============================================================

with gr.Blocks(
    title="Gerador BPMN"
) as demo:

    gr.Markdown("# 🤖 Gerador Automatizado de Grafos BPMN com Robozito")

    descricao = gr.Textbox(
        label="Descrição do Processo",
        lines=20,
        placeholder="""
Exemplo:
Processo de Solicitação de Equipamento
O colaborador registra uma solicitação de equipamento no sistema.
O setor de TI analisa a solicitação para verificar se as informações estão completas.
Caso as informações não estejam completas, o setor de TI solicita correção ao colaborador. Após a correção, a solicitação deve retornar para análise.
Caso as informações estejam completas, o setor de TI verifica se o equipamento está disponível em estoque.
Se o equipamento estiver disponível, o equipamento é separado para entrega.
Se o equipamento não estiver disponível, o setor de compras realiza a aquisição do equipamento.
Após a separação ou aquisição, o gestor deve aprovar a entrega.
Caso o gestor não aprove a entrega, a solicitação é encerrada.
Caso o gestor aprove a entrega, o equipamento é entregue ao colaborador.
Após a entrega ou encerramento da solicitação, o processo é finalizado.
"""
    )


    with gr.Row():
      analisar = gr.Button(
        "Gerar JSON BPMN",
        variant="primary"
      )
      btn_limpar = gr.Button(
          "🗑️ Limpar"
      )
    with gr.Row():
      json_saida = gr.Code(
    label="JSON BPMN",
    language="json",
    lines=25,
    min_width=700,
    visible=True,
    value='{\n  "Gerador": "Gerador"\n}'
)

    with gr.Row():

        btn_json = gr.Button(
            "⚡Baixar JSON",
            variant="primary"
        )

        btn_bpmn = gr.Button(
            "⚡Gerar BPMN",
            variant="primary"
        )

    arquivo_json = gr.File(
        label="Arquivo JSON"
    )

    arquivo_bpmn = gr.File(
        label="Arquivo BPMN"
    )

    analisar.click(
        gerar_json_bpmn,
        inputs=descricao,
        outputs=json_saida
    )

    btn_json.click(
        salvar_json,
        inputs=json_saida,
        outputs=arquivo_json
    )

    btn_bpmn.click(
        salvar_bpmn,
        inputs=json_saida,
        outputs=arquivo_bpmn
    )

    btn_limpar.click(
    fn=limpar,
    inputs=[],
    outputs=[
        descricao,
        json_saida,
        arquivo_json,
        arquivo_bpmn
    ]
)

# ============================================================
# EXECUÇÃO
# ============================================================

demo.launch(
    share=True,
    debug=True
)